# Food 3D Scanning & Nutrition Estimation Pipeline
### Kaggle Cloud GPU Benchmark (Phase 0 Audit & Controlled Fine-Tuning)

This notebook verifies the environment, executes baseline benchmarks, trains the food-specific fine-tuning models, and runs the comparative evaluation suite on a free Kaggle GPU.

### 1. Verify Allocated GPU

In [ ]:
!nvidia-smi

### 2. Verify Python Runtime & Hardware Environment

In [ ]:
import sys
!python -m src.utils.verify_env

### 3. Run Core Scientific Unit Tests
Validates taxonomy alias resolution, empirical bulk density registry bounds, synthetic geometry volume math, and analytical Gaussian error propagation.

In [ ]:
!python -m src.utils.run_unit_tests

### 4. Execute the End-to-End Baseline Benchmark
Runs MobileNetV3-Small on the calibrated plate test sample (`datasets/raw/sample_plate_apple.png`), recovers metric scale via the 24.0 cm plate diameter, and computes portion mass with 95% confidence intervals and USDA FDC macronutrients.

In [ ]:
!python -m src.evaluation.phase0_benchmark

### 5. Inspect Baseline Results

In [ ]:
import json
from PIL import Image
import matplotlib.pyplot as plt

# Display the calibrated test image
img_path = "datasets/raw/sample_plate_apple.png"
img = Image.open(img_path)

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title("Calibrated 24.0 cm Plate Test Sample with Ground-Truth Apple")
plt.axis("off")
plt.show()

# Load and display benchmark output
with open("outputs/baseline_results/phase0_baseline_results.json", "r") as f:
    res = json.load(f)

mass_data = res["nutrition_and_mass"]["estimated_mass_grams"]
nutrients = res["nutrition_and_mass"]["estimated_portion_nutrients"]

print(f"\n{'='*50}")
print(f"Identified Food: {res['nutrition_and_mass']['canonical_name']}")
print(f"Estimated Volume: {res['scale_and_volume']['estimated_volume_cm3']} cm³")
print(f"Estimated Mass: {mass_data['mean']} g (95% CI: [{mass_data['95_ci'][0]}, {mass_data['95_ci'][1]}] g)")
print(f"Estimated Energy: {nutrients['energy_kcal']['estimate']} kcal (± {nutrients['energy_kcal']['uncertainty_std']} kcal)")
print(f"{'='*50}")

### 6. Verify Controlled Food Datasets & Smoke-Test Isolation
Verifies data integrity and strict isolation between training, validation, and smoke-test splits.

In [ ]:
import json
with open('data_manifests/smoke_test_manifest.json') as f:
    smoke = json.load(f)
with open('data_manifests/finetune_dataset_manifest.json') as f:
    ft = json.load(f)

print(f"Finetune Dataset: {len(ft['images'])} images (Train: {ft['split_summary']['train']}, Val: {ft['split_summary']['val']})")
print(f"Smoke-Test Dataset: {len(smoke['images'])} images (Independent Diagnostic Split)")
overlap = set(i['sha256'] for i in smoke['images']).intersection(set(i['sha256'] for i in ft['images']))
print(f"Overlap between sets: {len(overlap)} (Strict data isolation verified)")

### 7. Run Controlled Food-Specific Fine-Tuning
Executes:
- **Experiment 1**: Freezes backbone and trains 5-class classifier head (15 epochs).
- **Experiment 2**: Staged fine-tuning with upper bottleneck blocks 11 and 12 unfrozen (10 epochs).

In [ ]:
!python -m src.classification.trainer

### 8. Run Three-Way Comparative Evaluation Benchmark
Compares:
1. Original Pretrained MobileNetV3-Small (ImageNet weights)
2. Experiment 1: Head-Only Checkpoint
3. Experiment 2: Staged Fine-Tuned Checkpoint
across both the validation split and the independent smoke-test set.

In [ ]:
!python -m src.evaluation.comparative_evaluation

### 9. Display Comparative Evaluation Summary

In [ ]:
import json
with open("outputs/comparative_evaluation_results.json", "r") as f:
    comp = json.load(f)

print("=" * 65)
print("VALIDATION SET RESULTS (10 images, 5 classes)")
print("=" * 65)
models_meta = [
    ("m0_original", "Original ImageNet-1k Baseline"),
    ("m1_head_only", "Experiment 1 (Head-Only 5-Class)"),
    ("m2_finetuned", "Experiment 2 (Staged Bottleneck Unfrozen)")
]
for m_key, label in models_meta:
    m = comp["validation_set_metrics"][m_key]
    print(f"{label:42s}: Accuracy = {m['accuracy']:.1%}")

print("\n" + "=" * 65)
print("INDEPENDENT SMOKE-TEST RESULTS (17 images)")
print("=" * 65)
for m_key, label in models_meta:
    m = comp["smoke_test_metrics"][m_key]
    print(f"{label:42s}: Accuracy = {m['accuracy']:.1%}")